# Chronos-2 Calibration Check on Sensex

Step 1: load data, run Chronos-2 forecasts on a rolling window, save results.

In [1]:
import pandas as pd
import numpy as np
import torch
from chronos import BaseChronosPipeline

## Load data

In [2]:
df = pd.read_csv("../data/sensex_clean.csv", parse_dates=["Date"])
df.tail()

,Date,Open,High,Low,Close
4049,2026-05-04,77257.27,77910.75,76939.54,77269.40
4050,2026-05-05,77103.72,77151.33,76515.03,77017.79
4051,2026-05-06,77424.36,78022.78,76773.25,77958.52
4052,2026-05-07,78339.24,78384.70,77713.21,77844.52
4053,2026-05-08,77631.94,77647.44,77146.43,77328.19


In [3]:
prices = df["Close"].values
prices.shape

(4054,)

## Load Chronos-2 model

This downloads the model the first time, then it's cached.

In [4]:
pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu",
    torch_dtype=torch.float32,
)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

## Try one forecast first

Before running the full rolling window, let's just do one forecast and look at what comes out.

In [5]:
context_len = 512
horizon = 30
quantile_levels = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

context = prices[-(context_len + horizon):-horizon]
actual = prices[-horizon:]

context_tensor = torch.tensor(context, dtype=torch.float32)

# chronos-2 wants a LIST of series, even if we only have one
quantiles, mean = pipeline.predict_quantiles(
    inputs=[context_tensor],
    prediction_length=horizon,
    quantile_levels=quantile_levels,
)

# quantiles is a list (one entry per series we gave it)
# we only gave one series, so we look at quantiles[0]
# its shape is (n_variates, horizon, num_quantiles) -> (1, 30, 9)
quantiles[0].shape

torch.Size([1, 30, 11])

In [6]:
# look at day 1 forecast, all quantile levels
quantiles[0][0, 0, :]

tensor([72436.8516, 72926.0703, 73415.2188, 73779.1484, 74027.8906, 74321.8516,
        74575.2656, 74880.1562, 75241.4297, 75875.6641, 76495.4844])

In [7]:
# compare to what actually happened on day 1
actual[0]

np.float64(72696.39)

If this looks fine (a range of numbers around the actual value), we move to the rolling window below.

## Rolling window forecast

We slide through the whole history: take 512 days, forecast next 30, move forward 10 days, repeat.
This gives many forecasts instead of just one, which we need to check calibration.

In [8]:
step = 10
start = context_len
end = len(prices) - horizon

all_forecasts = []
all_actuals = []
all_dates = []

for i in range(start, end, step):
    context = prices[i - context_len:i]
    actual = prices[i:i + horizon]

    context_tensor = torch.tensor(context, dtype=torch.float32)

    quantiles, mean = pipeline.predict_quantiles(
        inputs=[context_tensor],
        prediction_length=horizon,
        quantile_levels=quantile_levels,
    )

    # quantiles[0] shape is (1, horizon, num_quantiles), we drop the first dim
    q_values = quantiles[0][0].numpy()

    all_forecasts.append(q_values)
    all_actuals.append(actual)
    all_dates.append(df["Date"].iloc[i])

    if i % 100 == 0:
        print("done up to window", i, "of", end)

In [9]:
all_forecasts = np.array(all_forecasts)
all_actuals = np.array(all_actuals)

all_forecasts.shape, all_actuals.shape

((352, 30, 11), (352, 30))

## Save results for the next notebook

In [10]:
np.save("forecasts.npy", all_forecasts)
np.save("actuals.npy", all_actuals)
print("saved")

saved


                 SENSEX CSV
                     │
                     ↓
              ┌──────────────┐
              │ pandas DataFrame │
              └──────────────┘
                     │
                     ↓
              Close prices only
                     │
                     ↓
              4,054 observations
                     │
                     ↓
        ┌───────────────────────────┐
        │      Rolling window       │
        │                           │
        │  Previous 512 observations│
        │             │             │
        │             ↓             │
        │        Chronos-2          │
        │             │             │
        │             ↓             │
        │    Next 30 observations   │
        │             │             │
        │       11 quantiles        │
        └───────────────────────────┘
                     │
                     ↓
              Move forward 10
                     │
                     ↓
              Repeat 352 times
                     │
             ┌───────┴────────┐
             ↓                ↓
       Forecasts           Actuals
       (352,30,11)         (352,30)
             │                │
             └───────┬────────┘
                     ↓
               Save to disk
              forecasts.npy
               actuals.npy